# Verify Earth coverage and halo correction

Compare the corrected Earth run with the preserved original. Read-only inputs; results are written under `data/processed/analysis/`. Checks output hashes, unchanged grids/land cover, recovered elevations/features, masks and numerical differences. Uses the existing project environment.

**Workflow stage: preprocessing / data readiness.** Use this notebook to inspect coverage, investigate exclusions, or verify corrections before EDA. Its saved results and pinned inputs preserve the original investigation. Notebook 13 builds the two lunar terrain views; notebook 14 explores the corrected Earth run and those views. See the [workflow and acceptance checks](../README.md). Moving this notebook does not change when the original findings were made.


In [ ]:
import hashlib
import json
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'outline.md').exists())
OLD = ROOT/'data/processed/earth_terrain_landcover/20261028T014314564317Z_8a3e5d'
NEW = ROOT/'data/processed/earth_terrain_landcover/20261028T063142045472Z_9d6d3b'
OUT = ROOT/'data/processed/analysis'/('earth-coverage-fix-'+datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)


def load(path):
    with rasterio.open(path) as src:
        return src.read(1,masked=True).astype('float32').filled(np.nan)


def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()


records={}
for label,directory in [('old',OLD),('new',NEW)]:
    record=json.loads((directory/'provenance.json').read_text())
    assert record['status']=='complete'
    errors=[]
    for item in record['outputs']:
        path=directory/item['path']
        if not path.is_file() or path.stat().st_size!=item['bytes'] or digest(path)!=item['sha256']:
            errors.append(item['path'])
    assert not errors,errors
    records[label]={'run':str(directory.relative_to(ROOT)),'outputs_verified':len(record['outputs']),
                    'provenance_sha256':digest(directory/'provenance.json')}
    print(label,'hash checks passed:',len(record['outputs']),flush=True)

rows=[]
changes=[]
features=['elevation_m','relative_elevation_m','slope_deg','laplacian_per_m','local_elevation_std_m']
for region in ['northern_arizona','lanzarote','atacama_san_pedro']:
    old_folder=OLD/'earth'/region/'grid_30m'
    new_folder=NEW/'earth'/region/'grid_30m'
    old_mask=load(old_folder/'valid_input_mask.tif');new_mask=load(new_folder/'valid_input_mask.tif')
    inside=np.isfinite(old_mask)
    assert np.array_equal(inside,np.isfinite(new_mask))
    old_z=load(old_folder/'elevation_m.tif');new_z=load(new_folder/'elevation_m.tif')
    old_joint=inside.copy();new_joint=inside.copy()
    for feature in features:
        with rasterio.open(old_folder/(feature+'.tif')) as src, rasterio.open(new_folder/(feature+'.tif')) as dst:
            assert (src.shape,src.transform,src.crs)==(dst.shape,dst.transform,dst.crs)
        old=load(old_folder/(feature+'.tif'));new=load(new_folder/(feature+'.tif'))
        old_joint &= np.isfinite(old);new_joint &= np.isfinite(new)
        assert np.isnan(new[~inside]).all()
        common=np.isfinite(old)&np.isfinite(new)
        delta=np.abs(new[common].astype('float64')-old[common].astype('float64'))
        changes.append({'region':region,'layer':feature,'common_cells':int(common.sum()),
                        'absolute_difference_p99':float(np.quantile(delta,.99)),
                        'absolute_difference_max':float(delta.max())})
        if feature=='slope_deg':assert not ((new<0)|(new>90)).any()
        if feature=='local_elevation_std_m':assert not (new<0).any()
        if feature=='relative_elevation_m':assert abs(float(np.nanmedian(new)))<.001
    old_natural=load(old_folder/'natural_screen_mask.tif')
    new_natural=load(new_folder/'natural_screen_mask.tif')
    assert not ((old_natural==1)&(new_natural!=1)).any()
    for layer in ['bare_sparse_fraction','built_fraction','crop_fraction','water_fraction','source_coverage']:
        np.testing.assert_array_equal(load(old_folder/(layer+'.tif')),load(new_folder/(layer+'.tif')))
    old_gaps=inside&~np.isfinite(old_z)
    old_screen_gaps=(old_natural==1)&~old_joint
    assert np.isfinite(new_z[old_gaps]).all()
    assert new_joint[old_screen_gaps].all()
    assert new_joint[inside].all()
    row={'region':region,'old_missing_elevation_cells':int(old_gaps.sum()),
         'recovered_elevation_cells':int((old_gaps&np.isfinite(new_z)).sum()),
         'new_missing_elevation_cells':int((inside&~np.isfinite(new_z)).sum()),
         'old_screened_missing_features':int(old_screen_gaps.sum()),
         'recovered_old_screened_feature_cells':int((old_screen_gaps&new_joint).sum()),
         'new_screened_missing_features':int(((new_natural==1)&~new_joint).sum()),
         'new_joint_feature_fraction_inside_roi':float(new_joint[inside].mean()),
         'old_natural_area_km2':float((old_natural==1).sum()*.0009),
         'new_natural_area_km2':float((new_natural==1).sum()*.0009),
         'newly_screened_cells':int(((new_natural==1)&(old_natural!=1)).sum()),
         'new_invalid_landcover_coverage_cells':int((inside&(new_mask!=1)).sum())}
    rows.append(row)
    print(region,row,flush=True)
    del old_z,new_z,old,new,delta

pd.DataFrame(rows).to_csv(OUT/'recovery_by_region.csv',index=False)
pd.DataFrame(changes).to_csv(OUT/'changes_on_previously_valid_cells.csv',index=False)
result={'status':'complete','runs':records,'regions':rows,'numerical_changes':changes,
        'scope':'Full output hashes, original grid/ROI preservation, exact land-cover layer equality, all diagnosed gap recovery, masks and ranges. No independent source-accuracy or suitability validation.'}
(OUT/'verification.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print('Verification saved:',OUT,flush=True)


In [ ]:
# Locate material numerical changes and the separate land-cover coverage exclusions.
from rasterio.warp import transform

old_dir = OLD / "earth/northern_arizona"
new_dir = NEW / "earth/northern_arizona"
a = load(old_dir / "grid_30m/elevation_m.tif")
b = load(new_dir / "grid_30m/elevation_m.tif")
difference = np.abs(b-a)
rows, columns = np.where(difference > 1)
with rasterio.open(new_dir / "grid_30m/elevation_m.tif") as src:
    x, y = rasterio.transform.xy(src.transform, rows, columns)
    longitude, latitude = transform(src.crs, "EPSG:4326", x, y)
with rasterio.open(old_dir / "native_mosaic.tif") as src:
    bounds = src.bounds
longitude, latitude = np.asarray(longitude), np.asarray(latitude)
distance = np.minimum.reduce([(longitude-bounds.left)*111320*np.cos(np.deg2rad(latitude)),
                              (bounds.right-longitude)*111320*np.cos(np.deg2rad(latitude)),
                              (latitude-bounds.bottom)*110574, (bounds.top-latitude)*110574])
with rasterio.open(new_dir / "grid_30m/valid_input_mask.tif") as src:
    excluded_rows, excluded_columns = np.where(src.read(1) == 0)
    x, y = rasterio.transform.xy(src.transform, excluded_rows, excluded_columns)
    longitude, latitude = transform(src.crs, "EPSG:4326", x, y)
longitude, latitude = np.asarray(longitude), np.asarray(latitude)
landcover_distance = np.minimum((-111-longitude)*111320*np.cos(np.deg2rad(latitude)), (36-latitude)*110574)
followup = {"arizona_previously_valid_elevation_cells_changed_over_1m": len(rows),
            "maximum_change_m": float(np.nanmax(difference)),
            "max_approx_distance_to_old_mosaic_edge_m": float(distance.max()) if len(rows) else None,
            "landcover_excluded_cells": len(excluded_rows),
            "landcover_exclusions_max_approx_distance_to_north_east_tile_edge_m": float(landcover_distance.max()) if len(excluded_rows) else None}
(OUT / "boundary_followup.json").write_text(json.dumps(followup, indent=2), encoding="utf-8")
print(followup)

(OUT / "provenance.json").write_text(json.dumps({"status": "publishing", "verification": "verification.json", "runs": records}, indent=2), encoding="utf-8")


## Save reusable data and disposable reports
Run this final cell after the displays. Later notebooks read only the data directory; reports can be deleted and regenerated.

In [ ]:
import sys

sys.path.insert(0, str(ROOT / "notebooks"))
from analysis_artifacts import publish_analysis

REPORT_OUT = publish_analysis(ROOT, OUT, manifest_name="provenance.json")
print("Reusable data:", OUT)
print("Disposable reports:", REPORT_OUT)